In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from coord_converter import pixel_to_shelf

camera_matrix = np.load("camera_matrix.npy")
dist_coeffs = np.load("dist_coeffs.npy")

# Compute undistortion map
def init_undistort(frame_width, frame_height):
    new_cam_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dist_coeffs, (frame_width, frame_height), 1
    )
    map1, map2 = cv2.initUndistortRectifyMap(
        camera_matrix, dist_coeffs, None, new_cam_matrix,
        (frame_width, frame_height), cv2.CV_16SC2
    )
    return map1, map2


# Load your YOLO model
model = YOLO("model/runs/train/toy_animals_full/weights/best.pt")

# Grid configuration for virtual checkerboard
GRID_ROWS = 3
GRID_COLS = 2

def draw_virtual_grid(frame):
    """Draws a virtual 3x2 grid on the frame."""
    h, w = frame.shape[:2]
    cell_w = w // GRID_COLS
    cell_h = h // GRID_ROWS

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            x1 = c * cell_w
            y1 = r * cell_h
            x2 = x1 + cell_w
            y2 = y1 + cell_h

            # Draw box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 1)

            # Label cell coordinates
            text = f"({r}, {c})"
            cv2.putText(frame, text, (x1 + 10, y1 + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)

def detect_animals(frame):
    """YOLO detect animals and return list: (label, cx, cy)."""
    results = model(frame)[0]
    detections = []

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        detections.append((label, cx, cy))

    return detections

def get_grid_cell(cx, cy, frame_width, frame_height):
    """Return (row, col) of the grid cell where the point falls."""
    cell_w = frame_width // GRID_COLS
    cell_h = frame_height // GRID_ROWS

    col = cx // cell_w
    row = cy // cell_h

    # Clamp inside range
    row = min(max(row, 0), GRID_ROWS - 1)
    col = min(max(col, 0), GRID_COLS - 1)

    return int(row), int(col)

def main():
    cap = cv2.VideoCapture(1)
    print("Camera opened:", cap.isOpened())

    # Read one frame to initialize undistortion
    ret, frame = cap.read()
    h, w = frame.shape[:2]
    map1, map2 = init_undistort(w, h)

    print("Undistortion enabled. Starting system...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ---- UNDISTORT FRAME HERE ----
        frame = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)

        display = frame.copy()

        # Draw grid
        draw_virtual_grid(display)

        # YOLO detect on undistorted frame
        detections = detect_animals(frame)

        for label, cx, cy in detections:

            # Convert pixel → real coordinates
            X, Y, Z = pixel_to_shelf(cx, cy, shelf_z=0)

            print(f"{label} located at shelf coordinates: X={X:.2f}, Y={Y:.2f}, Z={Z:.2f}")

            # Draw detection
            cv2.circle(display, (cx, cy), 7, (0, 255, 0), -1)

            row, col = get_grid_cell(cx, cy, w, h)

            cv2.putText(display,
                f"{label} in cell ({row}, {col})",
                (cx - 50, cy - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 255, 255), 2,
            )

        cv2.imshow("Virtual Checkerboard Vision System", display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


In [ ]:
pip install ultralytics
